# Using Skills with Sec-Gemini SDK

This notebook demonstrates how to create, upload, use, and delete skills using the Sec-Gemini Python SDK.

In [ ]:
!pip install -q sec-gemini

## Authentication

Set your API key. Get one from [secgemini.google/keys](https://secgemini.google/keys).

In [ ]:
import os

# Option 1: From environment variable
API_KEY = os.environ.get("SEC_GEMINI_API_KEY")

# Option 2: From Colab secrets
if not API_KEY:
    try:
        from google.colab import userdata  # type: ignore[import-not-found]

        API_KEY = userdata.get("SEC_GEMINI_API_KEY")
    except (ImportError, Exception):
        pass

# Option 3: Paste directly (not recommended for shared notebooks)
if not API_KEY:
    API_KEY = "YOUR_API_KEY_HERE"

assert (
    API_KEY and API_KEY != "YOUR_API_KEY_HERE"
), "Please set your API key"

## Define and Upload a Skill

Skills are markdown files with YAML frontmatter. Here we define a simple skill inline and upload it.

In [ ]:
from sec_gemini import SecGemini

client = SecGemini(api_key=API_KEY)
await client.start()

skill_name = "hello_world_skill.yaml"
skill_content = """---
name: hello_world
description: A simple hello world skill
---
instructions: |
  Say hello to the user!
"""

print(f"Uploading skill '{skill_name}'...")
await client.skills.upload(name=skill_name, content=skill_content)
print("Successfully uploaded the skill.")

## Use the Skill

Create a session and prompt the agent. The agent should follow the instructions in the uploaded skill if it's relevant.

In [ ]:
session = await client.sessions.create()
print(f"Session created: {session.id}")

await session.prompt("Hi there!")

async for msg in session.messages.stream():
    print(f"\nMessage> Title: {msg.get('title')}")
    print(f"         Content: {msg.get('content')}")

## List and Delete Skills

You can list all uploaded skills and delete them when no longer needed.

In [ ]:
# List uploaded skills
print("\nListing uploaded skills:")
uploaded = await client.skills.list_uploaded()
for name in uploaded:
    print(f" - {name}")

# Delete the skill
print(f"\nDeleting skill '{skill_name}':")
await client.skills.delete(name=skill_name)
print("Done!")

## Cleanup

Clean up the session and close the client connection.

In [ ]:
await session.delete()
await client.close()
print("Client closed.")